In [1]:
"""
Model Training and Experiment Tracking
------------------------------------
This notebook:
- Loads cleaned heart disease data
- Performs feature engineering using sklearn Pipelines
- Trains Logistic Regression and Random Forest models
- Tracks experiments using MLflow
"""

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn

# Force MLflow to use the local tracking server
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Explicitly set experiment
mlflow.set_experiment("Heart_Disease_Classification")


2026/01/05 13:10:18 INFO mlflow.tracking.fluent: Experiment with name 'Heart_Disease_Classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1767598818307, experiment_id='1', last_update_time=1767598818307, lifecycle_stage='active', name='Heart_Disease_Classification', tags={}>

In [2]:
"""
Load the cleaned dataset produced in Phase 1.
This ensures strict separation between preprocessing and modeling.
"""

df = pd.read_csv("../data/heart_cleaned.csv")

df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [3]:
"""
Separate input features (X) and target variable (y).
"""

X = df.drop(columns=["target"])
y = df["target"]


In [4]:
"""
Split data into training and test sets.

Stratification ensures class balance is preserved
in both training and test sets.
"""

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [5]:
numerical_features = X.columns.tolist()

#Numerical preprocessing:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features)
    ]
)


In [6]:
log_reg_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [7]:
with mlflow.start_run(run_name="Logistic_Regression"):
    
    # Train model
    log_reg_pipeline.fit(X_train, y_train)
    
    # Predictions
    y_pred = log_reg_pipeline.predict(X_test)
    y_prob = log_reg_pipeline.predict_proba(X_test)[:, 1]
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    # Log parameters
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Log model artifact
    mlflow.sklearn.log_model(log_reg_pipeline, "model")
    
    print("Logistic Regression Metrics:")
    print(f"Accuracy : {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall   : {recall:.3f}")
    print(f"ROC-AUC  : {roc_auc:.3f}")


2026/01/05 13:10:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logistic Regression Metrics:
Accuracy : 0.869
Precision: 0.812
Recall   : 0.929
ROC-AUC  : 0.951
🏃 View run Logistic_Regression at: http://127.0.0.1:5000/#/experiments/1/runs/9ceddf9da4c5475d85e59c4d1fe8a6f6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Random Forest

In [8]:
"""
Random Forest:
- Captures non-linear relationships
- Robust to feature scaling
- Strong performance baseline
"""

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]
)


#### Train & Track Random Forest with MLflow

In [9]:
with mlflow.start_run(run_name="Random_Forest"):
    
    rf_pipeline.fit(X_train, y_train)
    
    y_pred = rf_pipeline.predict(X_test)
    y_prob = rf_pipeline.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("roc_auc", roc_auc)
    
    mlflow.sklearn.log_model(rf_pipeline, "model")
    
    print("Random Forest Metrics:")
    print(f"Accuracy : {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall   : {recall:.3f}")
    print(f"ROC-AUC  : {roc_auc:.3f}")


2026/01/05 13:10:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest Metrics:
Accuracy : 0.902
Precision: 0.844
Recall   : 0.964
ROC-AUC  : 0.955
🏃 View run Random_Forest at: http://127.0.0.1:5000/#/experiments/1/runs/b638e91cb7c149f7b498c99a10d1e8b9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### register the best model for inference

In [10]:
"""
Register the best-performing model in MLflow Model Registry.
"""

model_name = "HeartDiseaseClassifier"

mlflow.sklearn.log_model(
    rf_pipeline,
    artifact_path="model",
    registered_model_name=model_name
)


2026/01/05 13:21:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'HeartDiseaseClassifier'.
2026/01/05 13:21:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: HeartDiseaseClassifier, version 1
Created version '1' of model 'HeartDiseaseClassifier'.
